# Batch ETL и качество данных — 30 заданий

Практика выполняется на eBay в `/data/raw/ebay`. Решений нет.

## Результаты обучения

После **Batch ETL и качество** вы должны объяснить внутренний механизм, предсказать изменения metadata/files, выбрать безопасную команду и доказать итог измерением, а не сообщением об успехе.

## Архитектурная модель

Raw сохраняет вход, staging типизирует, accepted/reject объясняют судьбу строки, publish происходит после reconciliation. Watermark обновляется только после успешного batch.

```text
client ── metadata RPC ──► NameNode
  │                         │ block locations
  └── data stream ──► DataNode 1 ──► DataNode 2

HiveServer2 ──► Metastore (schema/location/partitions)
      └──────► execution engine ──► HDFS files
```
NameNode не хранит содержимое файла, а Metastore не хранит строки таблицы.

## Физическая схема eBay

```text
/data/raw/ebay/
├── snapshot_dt=2026-06-24/part-....snappy.parquet
├── snapshot_dt=2026-06-25/part-....snappy.parquet
└── ...
```
Grain: наблюдение `itemid` в `snapshot_dt`. Группы колонок: карточка/цена,
иерархия категорий, продавец, география и доставка. Полная schema — в `data-catalog`.

Общий raw read-only; результаты принадлежат `/user/$HDFS_USER/hadoop_training` и личной Hive DB.

## Алгоритм исследования

1. Зафиксируйте path/URI, owner и ожидаемый объект. 2. Снимите состояние до. 3. Выполните одно изменение. 4. Проверьте exit code. 5. Измерьте namespace/files/bytes/schema/rows. 6. Повторите команду и оцените идемпотентность. 7. Сохраните evidence.

Для каждой порции докажите read=accepted+rejected, key uniqueness, target delta и idempotent rerun.

## Типичные ошибки

- Путать локальный путь с HDFS URI.
- Делать вывод по `ls`, не проверяя blocks/bytes/schema.
- Использовать root или 777 вместо модели доступа.
- Создавать partition-каталог без Metastore или metadata без файлов.
- Считать replication резервной копией.
- Игнорировать малые файлы и цену NameNode metadata.

## Самопроверка

1. Какие metadata изменятся? 2. Где физически лежат bytes? 3. Сколько logical и physical bytes? 4. Кто может читать/писать? 5. Что произойдёт при повторе? 6. Какая независимая команда опровергнет вывод?

## Ментальная модель

Batch обязан быть идемпотентным и сверяемым: read = accepted + rejected, ключи уникальны, публикация атомарна. Watermark обновляют только после успешной порции.

## Подробная теория

### 1. Слои

Landing фиксирует вход, staging типизирует, core применяет бизнес-ключи, quality публикует доказательства. Raw не исправляют задним числом.

### 2. Контракты

Проверяйте обязательность, тип, диапазон, domain, уникальность и ссылочную целостность. Reject сохраняет причину, исходную строку и run_id.

### 3. Идемпотентность

Повтор одной порции не меняет итог. Обычно это overwrite партиции или MERGE по ключу, а не безусловный append.

### 4. Инкремент

Watermark хранит подтверждённую границу. Опоздавшие события требуют overlap/reprocessing; обновлять watermark до publish нельзя.

### 5. Reconciliation

read = accepted + rejected, target delta согласуется с accepted, агрегаты источника и цели совпадают в допустимой точности.

## Стенд

NameNode `namenode:8020`, два DataNode, HiveServer2 `hiveserver2:10000`. Личные артефакты не создаются в общем read-only raw-слое.

## Как сдаётся задание

Валидатор проверяет артефакт и JSON-доказательство. В `command` запишите фактическую команду, в `observation` — измеренный результат, в `explanation` — почему он получился. Минимальная длина защищает от пустых ответов; содержательный смысл остаётся вашей ответственностью.

In [ ]:
import json, os, subprocess, tempfile

def save_evidence(module, task, command, observation, explanation):
    user=os.environ.get("HDFS_USER", os.environ.get("HADOOP_USER_NAME", "student"))
    target=f"/user/{user}/hadoop_training/evidence/{module}/task_{task:02d}.json"
    payload={"module":module,"task":task,"command":command,"observation":observation,"explanation":explanation}
    with tempfile.NamedTemporaryFile("w",encoding="utf-8",delete=False,suffix=".json") as f:
        json.dump(payload,f,ensure_ascii=False,indent=2); local=f.name
    subprocess.run(["hdfs","dfs","-mkdir","-p",target.rsplit("/",1)[0]],check=True)
    subprocess.run(["hdfs","dfs","-put","-f",local,target],check=True)
    os.unlink(local)
    print(target)

### Задание 1. landing table

Создайте Hive-объект `dq_01` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `landing table`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 1

### Задание 2. staging table

Создайте Hive-объект `dq_02` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `staging table`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 2

### Задание 3. typed projection

Создайте Hive-объект `dq_03` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `typed projection`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 3

### Задание 4. required fields

Создайте Hive-объект `dq_04` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `required fields`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 4

### Задание 5. range check

Создайте Hive-объект `dq_05` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `range check`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 5

### Задание 6. domain check

Создайте Hive-объект `dq_06` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `domain check`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 6

### Задание 7. duplicate key

Создайте Hive-объект `dq_07` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `duplicate key`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 7

### Задание 8. reject view

Создайте Hive-объект `dq_08` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `reject view`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 8

### Задание 9. accepted view

Создайте Hive-объект `dq_09` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `accepted view`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 9

### Задание 10. row reconciliation

Создайте Hive-объект `dq_10` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `row reconciliation`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 10

### Задание 11. null profile

Создайте Hive-объект `dq_11` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `null profile`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 11

### Задание 12. distinct profile

Создайте Hive-объект `dq_12` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `distinct profile`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 12

### Задание 13. min max profile

Создайте Hive-объект `dq_13` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `min max profile`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 13

### Задание 14. partition completeness

Создайте Hive-объект `dq_14` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `partition completeness`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 14

### Задание 15. file completeness

Создайте Hive-объект `dq_15` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `file completeness`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 15

### Задание 16. schema validation

Создайте Hive-объект `dq_16` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `schema validation`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 16

### Задание 17. freshness

Создайте Hive-объект `dq_17` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `freshness`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 17

### Задание 18. idempotent overwrite

Создайте Hive-объект `dq_18` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `idempotent overwrite`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 18

### Задание 19. incremental append

Создайте Hive-объект `dq_19` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `incremental append`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 19

### Задание 20. watermark

Создайте Hive-объект `dq_20` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `watermark`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 20

### Задание 21. late data

Создайте Hive-объект `dq_21` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `late data`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 21

### Задание 22. dedup snapshot

Создайте Hive-объект `dq_22` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `dedup snapshot`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 22

### Задание 23. current record

Создайте Hive-объект `dq_23` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `current record`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 23

### Задание 24. history record

Создайте Hive-объект `dq_24` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `history record`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 24

### Задание 25. daily aggregate

Создайте Hive-объект `dq_25` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `daily aggregate`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 25

### Задание 26. quality summary

Создайте Hive-объект `dq_26` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `quality summary`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 26

### Задание 27. publish gate

Создайте Hive-объект `dq_27` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `publish gate`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 27

### Задание 28. rerun partition

Создайте Hive-объект `dq_28` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `rerun partition`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 28

### Задание 29. lineage columns

Создайте Hive-объект `dq_29` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `lineage columns`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 29

### Задание 30. batch pipeline

Создайте Hive-объект `dq_30` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `batch pipeline`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py etl_quality 30

## Итог

Все 30 проверок должны возвращать PASS. Удалять чужие или raw-данные запрещено.